# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR⁲](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a Dataset object, not a dictionary

print(f"Dataset Title: {metadata.name}")
print(f"Short Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets in the dataset by their @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- Name: {rs.name:40}  | @id: {rs.id}")

# For each record set, list fields by @id
for rs in record_sets:
    print(f"\nRecord set: {rs.name} (@id: {rs.id})")
    print("Fields:")
    for field in rs.fields:
        print(f" - {field.name:30} | @id: {field.id} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Assemble DataFrames for each record set by @id
dataframes = {}
for rs in record_sets:
    recs = list(dataset.records(record_set=rs.id))  # Use record_set @id
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for record set '{rs.name}' (@id: {rs.id})")
    print(f"First few columns: {df.columns[:5].tolist()}")
    print(df.head(2))

# For demonstration, select the first record set
main_record_set = record_sets[0]
df_main = dataframes[main_record_set.id]
print(f"\nAvailable columns in DataFrame for '{main_record_set.name}' (@id: {main_record_set.id}):\n{df_main.columns.tolist()}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing such as filtering, normalization, and grouping. Reference fields by their `@id` in code.

In [ ]:
import numpy as np

# List numeric fields in the main record set
numeric_fields = [f for f in main_record_set.fields if f.data_type in ['Float', 'Integer', 'Number']]
print("Numeric fields:")
for f in numeric_fields:
    print(f" - {f.name} (@id: {f.id})")

# Choose a sample numeric field by @id
if numeric_fields:
    numeric_field = numeric_fields[0].id
    print(f"Using numeric field '{numeric_fields[0].name}' with @id '{numeric_field}'")
else:
    print("No numeric fields found.")
    numeric_field = df_main.columns[0]  # fallback

# Filter records on the numeric field (> threshold)
threshold = np.nanquantile(df_main[numeric_field], 0.75)  # Example: filter upper 25%
filtered_df = df_main[df_main[numeric_field] > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (top 25%):")
print(filtered_df.head())

# Normalize
mean = filtered_df[numeric_field].mean()
std = filtered_df[numeric_field].std()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
categorical_fields = [f for f in main_record_set.fields if f.data_type=='Text']
if categorical_fields:
    group_field = categorical_fields[0].id
    print(f"Grouping by '{categorical_fields[0].name}' (@id: {group_field})")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()  # mean for demonstration
    print(grouped_df.head())
else:
    print("No categorical fields available for grouping.")

## 5. Visualization
Visualize a data distribution or relationship between fields. Fields are referenced by their `@id`.

Below we show a histogram of the main numeric field and a boxplot grouped by a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_main[numeric_field], kde=True, bins=20)
plt.title(f"Distribution of '{numeric_field}' in Record Set '{main_record_set.id}'")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by categorical group if available
if categorical_fields:
    group_field = categorical_fields[0].id
    plt.figure(figsize=(10,5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f"'{numeric_field}' grouped by '{group_field}'")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored a Croissant-described mass spectrometry dataset using `mlcroissant`. By referencing record set, field, and column entities via their `@id`, we:
- Inspected available record sets and fields in the dataset.
- Loaded the dataset and converted records to a DataFrame.
- Filtered, normalized, and grouped data using `@id`s.
- Visualized the distributions of key fields.

This approach ensures reproducibility and interoperability for FAIR data analytics. For further, domain-specific analysis, consult the Croissant JSON-LD schema and associated documentation.